<a href="https://colab.research.google.com/github/huy-V0/ai_research/blob/main/notebooks/refusal_direction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = "/content/drive/MyDrive/persona_rqa"
!mkdir -p {ROOT}/vectors {ROOT}/results {ROOT}/cache

In [ ]:
!pip install -q transformers==4.51.0 accelerate datasets

In [ ]:
import json, random
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 0
LAYER = 14
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
N_PER_SET = 64

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="cuda"
)
model.eval()

N_LAYERS = model.config.num_hidden_layers
D_MODEL = model.config.hidden_size
print(N_LAYERS, D_MODEL)
assert (N_LAYERS, D_MODEL) == (28, 1536)

In [ ]:
ADV_URL = ("https://raw.githubusercontent.com/llm-attacks/llm-attacks/"
           "main/data/advbench/harmful_behaviors.csv")

adv = pd.read_csv(ADV_URL)
print(len(adv), adv.columns.tolist())
harmful_pool = adv["goal"].tolist()
harmful = random.Random(SEED).sample(harmful_pool, N_PER_SET)
print(harmful[0])

In [ ]:
from datasets import load_dataset

alp = load_dataset("tatsu-lab/alpaca")["train"]
harmless_pool = [r["instruction"] for r in alp if r["input"].strip() == ""]
print(len(harmless_pool))
harmless = random.Random(SEED + 1).sample(harmless_pool, N_PER_SET)
print(harmless[0])

In [ ]:
prompt_spec = {
    "seed": SEED,
    "n_per_set": N_PER_SET,
    "harmful_source": "AdvBench harmful_behaviors.csv, column goal",
    "harmless_source": "tatsu-lab/alpaca, instruction where input is empty",
    "harmful": harmful,
    "harmless": harmless,
}
with open(f"{ROOT}/results/prompts_v1.json", "w") as f:
    json.dump(prompt_spec, f, indent=2)
print("saved prompts_v1.json")

In [ ]:
from tqdm.auto import tqdm

@torch.no_grad()
def get_resid_all(prompts):
    outs = []
    for p in tqdm(prompts):
        msgs = [{"role": "user", "content": p}]
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        ids = tok(text, return_tensors="pt").to("cuda")
        hs = model(**ids, output_hidden_states=True).hidden_states
        stacked = torch.stack([h[0, -1, :] for h in hs])   # [n_layers+1, d_model]
        outs.append(stacked.float().cpu())
    return torch.stack(outs)                                # [n, n_layers+1, d_model]


def get_resid(prompts, layer):
    return get_resid_all(prompts)[:, layer, :]

In [ ]:
A_harmful = get_resid_all(harmful)
A_harmless = get_resid_all(harmless)
print(A_harmful.shape, A_harmless.shape, A_harmful.dtype)
assert A_harmful.shape == (N_PER_SET, N_LAYERS + 1, D_MODEL)

In [ ]:
torch.save({"harmful": A_harmful, "harmless": A_harmless,
            "model": MODEL_NAME, "seed": SEED,
            "note": "last prompt token, all layers, float32"},
           f"{ROOT}/cache/acts_refusal_all_layers.pt")
!ls -lh {ROOT}/cache

In [ ]:
def direction_at(layer, A_h=A_harmful, A_l=A_harmless):
    d = A_h[:, layer, :].mean(0) - A_l[:, layer, :].mean(0)
    return d / d.norm()

r = direction_at(LAYER)
print(r.shape, r.dtype, float(r.norm()))

In [ ]:
payload = {
    "vector": r,
    "model": MODEL_NAME,
    "layer": LAYER,
    "layer_convention": "hidden_states[14], output of block 13",
    "token_position": "last prompt token, add_generation_prompt=True",
    "forward_dtype": "float16",
    "saved_dtype": "float32",
    "normalization": "unit L2",
    "n_harmful": N_PER_SET,
    "n_harmless": N_PER_SET,
    "seed": SEED,
    "prompts_file": "results/prompts_v1.json",
}
torch.save(payload, f"{ROOT}/vectors/refusal_L{LAYER}.pt")
print("saved refusal_L%d.pt" % LAYER)

In [ ]:
# Separation at layer 14: project both sets onto r and compare.
from sklearn.metrics import roc_auc_score

def separation(layer):
    d = direction_at(layer)
    ph = A_harmful[:, layer, :] @ d
    pl = A_harmless[:, layer, :] @ d
    y = np.r_[np.ones(len(ph)), np.zeros(len(pl))]
    s = np.r_[ph.numpy(), pl.numpy()]
    return roc_auc_score(y, s), float(ph.mean()), float(pl.mean())

auc, mh, ml = separation(LAYER)
print("layer %d  AUC %.3f  mean harmful %.2f  mean harmless %.2f" % (LAYER, auc, mh, ml))

In [ ]:
# Layer sweep. In-sample AUC, so treat it as a smoke test, not a selection rule.
for L in range(1, N_LAYERS + 1):
    a, _, _ = separation(L)
    print("%2d  %.3f  %s" % (L, a, "#" * int(a * 40)))

In [ ]:
# Cosine between the layer-14 direction and its neighbours.
for L in [10, 12, 13, 15, 16, 20]:
    print(L, round(float(direction_at(L) @ r), 3))